# Task 4: Image Classification Model

**Goal:** Implement an image classification model that can classify images into different object categories.

In this notebook, we will classify images into three classes:

- bird
- cat
- dog

We will use the **CIFAR-10** dataset because it already contains these image categories.  
We will filter only bird, cat, and dog images from CIFAR-10 and train a CNN model.

**Main tools used:**
- TensorFlow / Keras
- NumPy
- Matplotlib
- PIL for loading custom images

In [ ]:
# Install TensorFlow if it is not already installed.
# In Google Colab, TensorFlow is usually already installed.
!pip install tensorflow matplotlib pillow

## Step 1: Import libraries

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow.keras.datasets import cifar10
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization
from tensorflow.keras.utils import to_categorical
from PIL import Image

print("TensorFlow Version:", tf.__version__)

## Step 2: Load CIFAR-10 dataset

CIFAR-10 has 10 classes.  
We will keep only these 3 classes:

- bird
- cat
- dog

In [ ]:
# Load CIFAR-10 dataset
(x_train, y_train), (x_test, y_test) = cifar10.load_data()

# CIFAR-10 class index mapping:
# 0 airplane, 1 automobile, 2 bird, 3 cat, 4 deer,
# 5 dog, 6 frog, 7 horse, 8 ship, 9 truck

selected_classes = {
    2: "bird",
    3: "cat",
    5: "dog"
}

# New labels:
# bird = 0, cat = 1, dog = 2
new_label_map = {
    2: 0,
    3: 1,
    5: 2
}

class_names = ["bird", "cat", "dog"]

print("Original training data shape:", x_train.shape)
print("Original testing data shape:", x_test.shape)

## Step 3: Filter bird, cat, and dog images

In [ ]:
def filter_selected_classes(images, labels, selected_class_ids, new_label_mapping):
    """
    This function filters only selected classes from the dataset.
    It also converts original CIFAR-10 labels into new labels.
    """
    filtered_images = []
    filtered_labels = []

    for img, label in zip(images, labels.flatten()):
        if label in selected_class_ids:
            filtered_images.append(img)
            filtered_labels.append(new_label_mapping[label])

    return np.array(filtered_images), np.array(filtered_labels)

# Filter training and testing data
x_train_filtered, y_train_filtered = filter_selected_classes(
    x_train, y_train, selected_classes.keys(), new_label_map
)

x_test_filtered, y_test_filtered = filter_selected_classes(
    x_test, y_test, selected_classes.keys(), new_label_map
)

print("Filtered training data shape:", x_train_filtered.shape)
print("Filtered testing data shape:", x_test_filtered.shape)
print("Filtered classes:", class_names)

## Step 4: Normalize images and one-hot encode labels

In [ ]:
# Normalize image pixel values from range 0-255 to range 0-1
x_train_filtered = x_train_filtered.astype("float32") / 255.0
x_test_filtered = x_test_filtered.astype("float32") / 255.0

# Convert labels into one-hot encoded form
# Example: cat class 1 becomes [0, 1, 0]
y_train_encoded = to_categorical(y_train_filtered, num_classes=3)
y_test_encoded = to_categorical(y_test_filtered, num_classes=3)

print("One-hot label example:", y_train_encoded[0])

## Step 5: Display sample images

In [ ]:
plt.figure(figsize=(10, 4))

for i in range(12):
    plt.subplot(3, 4, i + 1)
    plt.imshow(x_train_filtered[i])
    plt.title(class_names[y_train_filtered[i]])
    plt.axis("off")

plt.tight_layout()
plt.show()

## Step 6: Build CNN model

CNN is suitable for image classification because it can learn patterns such as edges, shapes, and object parts.

In [ ]:
model = Sequential([
    # First convolution block
    Conv2D(32, (3, 3), activation="relu", padding="same", input_shape=(32, 32, 3)),
    BatchNormalization(),
    Conv2D(32, (3, 3), activation="relu", padding="same"),
    MaxPooling2D((2, 2)),
    Dropout(0.25),

    # Second convolution block
    Conv2D(64, (3, 3), activation="relu", padding="same"),
    BatchNormalization(),
    Conv2D(64, (3, 3), activation="relu", padding="same"),
    MaxPooling2D((2, 2)),
    Dropout(0.25),

    # Third convolution block
    Conv2D(128, (3, 3), activation="relu", padding="same"),
    BatchNormalization(),
    MaxPooling2D((2, 2)),
    Dropout(0.30),

    # Fully connected layers
    Flatten(),
    Dense(128, activation="relu"),
    Dropout(0.40),
    Dense(3, activation="softmax")  # 3 output classes: bird, cat, dog
])

model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

## Step 7: Train the model

In [ ]:
# Train the model.
# You can increase epochs for better accuracy.
history = model.fit(
    x_train_filtered,
    y_train_encoded,
    epochs=10,
    batch_size=64,
    validation_split=0.2,
    verbose=1
)

## Step 8: Plot training accuracy and loss

In [ ]:
# Plot accuracy
plt.figure(figsize=(7, 5))
plt.plot(history.history["accuracy"], label="Training Accuracy")
plt.plot(history.history["val_accuracy"], label="Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Training and Validation Accuracy")
plt.legend()
plt.show()

# Plot loss
plt.figure(figsize=(7, 5))
plt.plot(history.history["loss"], label="Training Loss")
plt.plot(history.history["val_loss"], label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training and Validation Loss")
plt.legend()
plt.show()

## Step 9: Evaluate the model

In [ ]:
test_loss, test_accuracy = model.evaluate(x_test_filtered, y_test_encoded, verbose=0)

print("Test Loss:", round(test_loss, 4))
print("Test Accuracy:", round(test_accuracy, 4))

## Step 10: Predict class for test images

In [ ]:
# Predict probabilities for test images
predictions = model.predict(x_test_filtered[:12])

plt.figure(figsize=(10, 4))

for i in range(12):
    predicted_class = np.argmax(predictions[i])
    actual_class = y_test_filtered[i]

    plt.subplot(3, 4, i + 1)
    plt.imshow(x_test_filtered[i])
    plt.title(f"Pred: {class_names[predicted_class]}\nActual: {class_names[actual_class]}")
    plt.axis("off")

plt.tight_layout()
plt.show()

## Step 11: Save the trained model

In [ ]:
# Save the model for future use
model.save("bird_cat_dog_classifier.h5")

print("Model saved successfully as bird_cat_dog_classifier.h5")

## Step 12: Predict a new image

Upload any bird, cat, or dog image and give its file path in the code below.

In Google Colab, you can use:

```python
from google.colab import files
uploaded = files.upload()
```

Then copy the uploaded image filename into `new_image_path`.

In [ ]:
def predict_new_image(new_image_path):
    """
    This function loads a new image, resizes it to 32x32,
    normalizes it, and predicts whether it is bird, cat, or dog.
    """
    img = Image.open(new_image_path).convert("RGB")
    img_resized = img.resize((32, 32))

    img_array = np.array(img_resized).astype("float32") / 255.0
    img_array = np.expand_dims(img_array, axis=0)

    prediction = model.predict(img_array)
    predicted_index = np.argmax(prediction)
    confidence = np.max(prediction)

    plt.imshow(img)
    plt.axis("off")
    plt.title(f"Predicted: {class_names[predicted_index]} | Confidence: {confidence:.2f}")
    plt.show()

    return class_names[predicted_index], confidence

# Example:
# Change this path to your uploaded image path.
# predicted_class, confidence = predict_new_image("my_test_image.jpg")

## Conclusion

This notebook trained a CNN model for image classification using bird, cat, and dog images.
The model can also predict the class of a new image after training.